# Row 4: the matcher decides when to ask, the decider decides whether it is the moment

What this tests and why: `docs/RATIONALE.md` (section 4, row 4; decisions D13, D20). It reuses row 3's stored similarities and row 1's
decider, so you do **not** re-run either. Only the decider runs, and only at the moments the matcher fires (about 1,800 calls on val,
roughly 10 minutes). Run the cells in order from the repo root.

A response needs both: the matcher fired **and** the decider said Yes. So this can only remove responses relative to the matcher alone;
the question is whether it removes more false alarms than true events.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":      # run from the repo root
    os.chdir("..")
print(os.getcwd())
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

c:\Users\abhin\Code\CV_Proj
torch 2.6.0+cu124 | cuda: True


## 1. Find the two caches

Row 3's similarities (matcher cache) and the decider cache. If either line prints nothing, run row 3 / row 1 first.

In [2]:
import glob
MATCHER = sorted(glob.glob("results/cache/matcher/Ops-MM-embedding-v1-2B__*"))[-1]
print("matcher:", MATCHER)
print("decider:", sorted(glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_only__*")))

matcher: results/cache/matcher\Ops-MM-embedding-v1-2B__875248e9
decider: ['results/cache/decider\\Qwen3-VL-2B-Instruct__query_only__1a7df950']


## 2. Smoke test: three questions

In [3]:
!python scripts/run_gated_decider.py --samples data/processed/ovo_far_val.jsonl --matcher {MATCHER} --limit 3

decider cache: results\cache\decider\Qwen3-VL-2B-Instruct__query_only__1a7df950
3 questions, 46 trigger moments to ask about (rule code, any cue, theta >= 0.02)
[1/3] ovo_CRR_1479 CRR: 22 moments, 19 new decider calls
[2/3] ovo_CRR_1480 CRR: 9 moments, 6 new decider calls
[3/3] ovo_CRR_1481 CRR: 15 moments, 9 new decider calls
done: 34 decider calls in 12 s



Loading weights: 100%|██████████| 625/625 [00:02<00:00, 244.76it/s]
c:\Users\abhin\Code\CV_Proj\.venv\Lib\site-packages\transformers\models\qwen2_vl\image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


## 3. Speed check

Median milliseconds per on-demand decider call (row 1 measured about 300).

In [4]:
import glob, json, statistics
rows = [json.loads(l) for f in glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_only__*/ondemand/*.jsonl") for l in open(f) if l.strip()]
print(len(rows), "on-demand calls; median", round(statistics.median(r["ms"] for r in rows)), "ms")

34 on-demand calls; median 216 ms


## 4. All val questions (resumable; never repeats a decider call)

In [5]:
!python scripts/run_gated_decider.py --samples data/processed/ovo_far_val.jsonl --matcher {MATCHER}

decider cache: results\cache\decider\Qwen3-VL-2B-Instruct__query_only__1a7df950
43 questions, 1841 trigger moments to ask about (rule code, any cue, theta >= 0.02)
[1/43] ovo_CRR_1479 CRR: 22 moments, 0 new decider calls
[2/43] ovo_CRR_1480 CRR: 9 moments, 0 new decider calls
[3/43] ovo_CRR_1481 CRR: 15 moments, 0 new decider calls
[4/43] ovo_CRR_1482 CRR: 31 moments, 24 new decider calls
[5/43] ovo_CRR_1483 CRR: 64 moments, 48 new decider calls
[6/43] ovo_CRR_1484 CRR: 18 moments, 14 new decider calls
[7/43] ovo_CRR_1485 CRR: 16 moments, 12 new decider calls
[8/43] ovo_SSR_1518 SSR: 78 moments, 59 new decider calls
[9/43] ovo_SSR_1519 SSR: 103 moments, 77 new decider calls
[10/43] ovo_SSR_1531 SSR: 61 moments, 44 new decider calls
[11/43] ovo_SSR_1533 SSR: 103 moments, 76 new decider calls
[12/43] ovo_SSR_1538 SSR: 296 moments, 231 new decider calls
[13/43] ovo_SSR_1540 SSR: 42 moments, 31 new decider calls
[14/43] ovo_SSR_1545 SSR: 97 moments, 72 new decider calls
[15/43] ovo_SSR_154


Loading weights: 100%|██████████| 625/625 [00:01<00:00, 364.31it/s]
c:\Users\abhin\Code\CV_Proj\.venv\Lib\site-packages\transformers\models\qwen2_vl\image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


## 5. Score it

Rows are matcher thresholds and cooldowns; columns are how sure the decider must be (tau 0.00 means the matcher alone, nothing removed).
Compare the best cell with row 1 (about 30 at similar response counts) and row 3 (about 30).

In [6]:
import glob
DECIDER = sorted(glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_only__*"))[-1]
!python scripts/replay_gated.py --matcher {MATCHER} --decider {DECIDER} --samples data/processed/ovo_far_val.jsonl

ovo_far_val: 43/43 questions with matcher similarities; 3378 stored decider answers
tau = 0.00 means the matcher alone (every trigger counts); the decider only removes responses.

 theta cooldown tau=0.00     tau=0.30     tau=0.50     tau=0.60     tau=0.70     tau=0.80     tau=0.90      avg F1 (responses)
  0.02      0.0    16.1(1841)    16.0(1703)    15.4(1541)    15.7(1384)    16.8(1146)    17.2( 657)    11.5( 103)
  0.02      4.0    29.4( 539)    28.4( 508)    26.9( 472)    25.8( 423)    28.5( 367)    27.8( 243)    13.9(  60)
  0.02      8.0    25.0( 348)    24.4( 331)    24.7( 308)    24.7( 277)    27.0( 247)    26.8( 172)    14.2(  48)
  0.03      0.0    22.3(1228)    21.9(1133)    20.7(1031)    20.8( 933)    21.5( 776)    19.0( 442)    12.0(  60)
  0.03      4.0    31.8( 426)    30.1( 403)    28.2( 377)    28.2( 342)    28.7( 299)    26.2( 193)    12.6(  43)
  0.03      8.0    27.4( 287)    29.3( 274)    27.0( 255)    26.5( 232)    26.3( 206)    23.7( 144)    12.8(  35)
  0.04   

## 6. Push the results

```
git add results/cache && git commit -m "Row 4 on-demand decider answers and replay, val" && git pull --rebase origin abhi && git push origin abhi
```
Please also push `results/cache/matcher` and `results/cache/cues` if you have not: the small `.json` files are what let per-question and per-task breakdowns be computed.